# 01 — Inspección Inicial

Este notebook documenta la estructura general del dataset `streaming_users_dirty.json` sin realizar modificaciones. El objetivo es reunir evidencia para orientar las etapas posteriores.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)

# Carga del dataset original (sin modificar)
data_path = Path("../data/raw/streaming_users_dirty.json")
if not data_path.exists():
    data_path = Path("data/raw/streaming_users_dirty.json")

df = pd.read_json(data_path)

print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (584324392.py, line 7)

## Estructura general

In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8160 entries, 0 to 8159
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8160 non-null   int64  
 1   age                       8160 non-null   int64  
 2   subscription_plan         8160 non-null   object 
 3   monthly_watch_time_mins   7967 non-null   float64
 4   country                   8160 non-null   object 
 5   favorite_genre            7920 non-null   object 
 6   last_login_date           7840 non-null   object 
 7   customer_support_tickets  8160 non-null   int64  
dtypes: float64(1), int64(3), object(4)
memory usage: 510.1+ KB


In [ ]:
df.head(10)


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1
5,10005,20,Básico,670.2,Uruguay,Drama,2020-07-03,2
6,10006,37,Básico,346.6,Perú,Thriller,2019-07-26,1
7,10007,31,Estándar,974.6,Chile,Acción,2019-02-24,1
8,10008,36,Premium,1432.2,Colombia,Romance,2025-08-03,2
9,10009,37,Estándar,1375.4,Argentina,Thriller,2024-02-12,1


## Tipos de datos

In [ ]:
print("Tipos de datos:")
print(df.dtypes)


Tipos de datos:
user_id                       int64
age                           int64
subscription_plan            object
monthly_watch_time_mins     float64
country                      object
favorite_genre               object
last_login_date              object
customer_support_tickets      int64
dtype: object


## Valores faltantes

In [ ]:
nulos = df.isnull().sum()
pct = (nulos / len(df) * 100).round(2)
resumen_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': pct})
print(resumen_nulos)


                          Nulos  Porcentaje (%)
user_id                       0            0.00
age                           0            0.00
subscription_plan             0            0.00
monthly_watch_time_mins     193            2.37
country                       0            0.00
favorite_genre              240            2.94
last_login_date             320            3.92
customer_support_tickets      0            0.00


## Duplicados

In [ ]:
dup = df.duplicated().sum()
print(f"Filas duplicadas: {dup} ({dup/len(df)*100:.2f}% del total)")


Filas duplicadas: 126 (1.54% del total)


## Valores únicos por variable categórica

Se detectan inconsistencias ortográficas y de formato en `subscription_plan`, `country` y `favorite_genre`.

In [ ]:
for col in ['subscription_plan', 'country', 'favorite_genre']:
    print(f"\n{'='*50}")
    print(f"--- {col} ({df[col].nunique(dropna=False)} valores únicos) ---")
    print(df[col].value_counts(dropna=False).to_string())


--- subscription_plan (15 valores únicos) ---
subscription_plan
Básico       3450
Estándar     2711
Premium      1519
basico         60
BASICO         52
Basic          52
básico         50
Std            48
Estándar       46
estandar       36
STANDARD       34
Premium        31
PREMIUM        26
Premiun        23
premium        22

--- country (26 valores únicos) ---
country
Brasil        1132
Chile         1132
México        1129
Uruguay       1124
Perú          1120
Colombia      1116
Argentina     1087
colombia        27
méxico          25
uruguay         24
Brazil          21
COL             19
CHL             18
URY             17
argentina       16
MEX             16
Chile           16
PER             16
Peru            15
BRA             15
chile           15
Mexico          15
brasil          13
perú            12
ARG             10
Argentina       10

--- favorite_genre (29 valores únicos) ---
favorite_genre
Comedia        1112
Drama          1105
Documental     1095
Romance

## Estadísticas descriptivas — variables numéricas

In [ ]:
df.describe().round(2)


,user_id,age,monthly_watch_time_mins,customer_support_tickets
count,8160.00,8160.00,7967.00,8160.00
mean,13995.43,34.10,1107.35,1.80
std,2310.81,14.51,5310.44,11.33
min,10000.00,-5.00,-120.00,-1.00
25%,11987.75,25.00,489.20,0.00
50%,13998.50,33.00,757.40,1.00
75%,15997.25,42.00,1045.70,1.00
max,17999.00,150.00,99999.00,150.00


## Observaciones iniciales

- **`age`**: valores negativos y mayores a 100 → posibles errores de carga.
- **`monthly_watch_time_mins`**: valores negativos y extremadamente altos → outliers a evaluar.
- **`customer_support_tickets`**: valores negativos y muy altos (hasta 150) → probable ruido.
- **`subscription_plan`**: múltiples variantes de texto para el mismo plan (Std, estandar, STANDARD, etc.).
- **`country`**: códigos ISO mezclados con nombres en distintos idiomas y capitalización.
- **`favorite_genre`**: mezcla de idiomas (comedy/Comedia, Documentary/Documental) y mayúsculas.
- **`last_login_date`**:
320 nulos detectados. El campo llega como string; su validez como fecha (formato, rango) se analizará en la etapa de limpieza
- **Duplicados**: 126 filas duplicadas detectadas.

## Preguntas de análisis

1. ¿Existe relación entre el plan de suscripción y el tiempo mensual de visualización?
2. ¿Cuál es la distribución de usuarios por país y género favorito?
3. ¿Qué variables numéricas presentan mayor correlación entre sí?
4. ¿Qué estructura latente revela el PCA sobre las variables numéricas?
5. ¿Los usuarios con más tickets de soporte tienen menor tiempo de visualización?